# Job Dimension (SCD Type 2) Loader

This notebook maintains the `warehouse.dim_job` dimension table with SCD Type 2 logic.

**Purpose**: Track job posting changes over time with effective date windows

**Key Features**:
* SCD Type 2 for tracking historical changes
* Stable surrogate keys (job_sk) for each version
* Enterprise job ID mapping via identity map
* Foreign key resolution to company, location, sector, role dimensions
* Effective date windows (effective_from, effective_to, is_current)

**Sources**: 
* `silver.silver_jobs_current` (job data)
* `silver.silver_job_identity_map` (canonical job IDs)
* Dimension tables for FK resolution

**Target**: `workspace.warehouse.dim_job`

In [0]:
%sql
-- Extract jobs with canonical identity mapping
CREATE OR REPLACE TEMP VIEW job_extract AS
SELECT 
  j.enterprise_job_id,
  j.source_name,
  j.source_job_id,
  j.company_name_norm,
  j.title_normalized,
  j.description_raw,
  j.location_norm,
  j.remote_type,
  j.salary_min,
  j.salary_max,
  j.salary_currency,
  j.employment_type_normalized,
  j.posted_at,
  j.last_seen,
  j.is_active,
  j.record_hash,
  j.updated_at
FROM workspace.silver.silver_jobs_current j
WHERE j.enterprise_job_id IS NOT NULL
  AND j.is_active = TRUE
  AND j.soft_delete_flag = FALSE;

In [0]:
%sql
-- Resolve foreign keys to all dimension tables
CREATE OR REPLACE TEMP VIEW job_with_fks AS
SELECT 
  j.*,
  -- Company FK (via canonical company mapping)
  COALESCE(dc.company_sk, -1) as company_sk,
  -- Location FK
  COALESCE(dl.location_sk, -1) as location_sk,
  -- Sector FK (via company)
  COALESCE(ds.sector_sk, -1) as sector_sk,
  -- Role FK (via semantic job role mapping)
  r.canonical_role_id,
  COALESCE(dr.role_sk, -1) as role_sk
FROM job_extract j
-- Join to company dimension via company alias
LEFT JOIN workspace.warehouse.dim_company_alias ca
  ON j.company_name_norm = ca.alias_name
LEFT JOIN workspace.warehouse.dim_company dc
  ON ca.company_sk = dc.company_sk
-- Join to location dimension
LEFT JOIN workspace.warehouse.dim_location dl
  ON j.location_norm = dl.location_name
-- Join to sector dimension (via company)
LEFT JOIN workspace.warehouse.dim_sector ds
  ON dc.sector_sk = ds.sector_sk
-- Join to role dimension (via semantic role mapping)
LEFT JOIN workspace.semantic.sem_job_role_map r
  ON j.title_normalized = r.title_normalized
LEFT JOIN workspace.warehouse.dim_role dr
  ON r.canonical_role_id = dr.canonical_role_id;

In [0]:
%sql
-- Detect changes and create SCD2 versions
CREATE OR REPLACE TEMP VIEW job_changes AS
SELECT 
  j.*,
  d.job_sk as existing_job_sk,
  d.record_hash as existing_hash,
  d.is_current as existing_is_current,
  CASE 
    WHEN d.job_sk IS NULL THEN 'INSERT'
    WHEN d.record_hash != j.record_hash AND d.is_current = TRUE THEN 'UPDATE'
    ELSE 'NOCHANGE'
  END as change_type
FROM job_with_fks j
LEFT JOIN workspace.warehouse.dim_job d
  ON j.enterprise_job_id = d.enterprise_job_id
  AND d.is_current = TRUE;

In [0]:
%sql
-- Close out existing current records that have changed
MERGE INTO workspace.warehouse.dim_job target
USING (
  SELECT DISTINCT
    existing_job_sk as job_sk,
    updated_at as effective_to
  FROM job_changes
  WHERE change_type = 'UPDATE'
    AND existing_job_sk IS NOT NULL
) source
ON target.job_sk = source.job_sk
  AND target.is_current = TRUE
WHEN MATCHED THEN UPDATE SET
  target.effective_to = source.effective_to,
  target.is_current = FALSE;

In [0]:
%sql
-- Insert new records and new versions of updated records
INSERT INTO workspace.warehouse.dim_job (
  job_sk,
  enterprise_job_id,
  canonical_role_id,
  company_sk,
  location_sk,
  sector_sk,
  title_normalized,
  salary_min,
  salary_max,
  salary_currency,
  remote_type,
  employment_type_normalized,
  effective_from,
  effective_to,
  is_current,
  record_hash
)
SELECT
  -- Generate new surrogate key
  ROW_NUMBER() OVER (ORDER BY enterprise_job_id, updated_at) + 
    COALESCE((SELECT MAX(job_sk) FROM workspace.warehouse.dim_job), 0) as job_sk,
  enterprise_job_id,
  canonical_role_id,
  company_sk,
  location_sk,
  sector_sk,
  title_normalized,
  salary_min,
  salary_max,
  salary_currency,
  remote_type,
  employment_type_normalized,
  updated_at as effective_from,
  NULL as effective_to,
  TRUE as is_current,
  record_hash
FROM job_changes
WHERE change_type IN ('INSERT', 'UPDATE');

In [0]:
%sql
-- Validate job dimension
SELECT 
  COUNT(*) as total_job_versions,
  COUNT(DISTINCT enterprise_job_id) as unique_jobs,
  SUM(CASE WHEN is_current THEN 1 ELSE 0 END) as current_versions,
  COUNT(*) - SUM(CASE WHEN is_current THEN 1 ELSE 0 END) as historical_versions,
  AVG(CASE WHEN effective_to IS NULL THEN 0 ELSE DATEDIFF(effective_to, effective_from) END) as avg_version_duration_days
FROM workspace.warehouse.dim_job;

-- Sample job history
SELECT 
  job_sk,
  enterprise_job_id,
  title_normalized,
  company_sk,
  effective_from,
  effective_to,
  is_current
FROM workspace.warehouse.dim_job
WHERE enterprise_job_id IN (
  SELECT enterprise_job_id 
  FROM workspace.warehouse.dim_job 
  GROUP BY enterprise_job_id 
  HAVING COUNT(*) > 1 
  LIMIT 5
)
ORDER BY enterprise_job_id, effective_from;